# Model key attribution

HumanEval experiment: Qwen 1.5B, Qwen 0.5B, and DeepSeek 1.3B, each assigned a different key. All 164 HumanEval problems per model, clean and watermarked outputs; bias 2, float32. Results pending execution.

## Environment

Local CPU execution, float32, fixed model revisions. Results remain on this laptop.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
from IPython.display import Image, Markdown, display

READY = False
PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / 'src/codeguard').is_dir() and (p / 'pyproject.toml').is_file()), None)
if PROJECT is None:
    raise RuntimeError('Open this notebook inside the local CodeGuardLLM project.')
os.chdir(PROJECT)
ENV = PROJECT / '.venv'
PYTHON = str(ENV / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python'))
if not Path(PYTHON).is_file():
    raise RuntimeError('Create the project .venv first; see README local installation.')
CONFIG = PROJECT / 'configs/local.yaml'
RUN_DIR = PROJECT / 'runs/local/humaneval/attribution'
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Project:', PROJECT)
print('Python:', PYTHON)
print('Results:', RUN_DIR)


In [ ]:
from collections import deque
from datetime import datetime, timezone
import os, subprocess

LOG_DIR = PROJECT / '.tmp/notebook-logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

def command(args, label='command'):
    log = LOG_DIR / (label + '-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f') + '.log')
    tail = deque(maxlen=30)
    print(f'Running {label}. Log: {log}', flush=True)
    with log.open('w', encoding='utf-8') as handle:
        with subprocess.Popen(
            list(map(str, args)), cwd=PROJECT, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace',
            bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'},
        ) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    handle.write(line)
                    handle.flush()
                    tail.append(line)
                status = process.wait()
            except KeyboardInterrupt:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
    if status:
        detail = ''.join(tail)
        raise RuntimeError(f'{label} failed (exit {status}). Full log: {log}\n\n{detail}')

def run(*args, label='codeguard'):
    command([PYTHON, '-u', *args], label=label)

In [ ]:
READY = False
run('-m', 'pip', 'install', '-e', '.[ml,data,analysis,notebook]', label='dependencies')
run('scripts/bootstrap_markllm.py', label='markllm')
run('-m', 'codeguard', 'validate', '--config', CONFIG, '--check-markllm', label='validate-local')
run('-c', "import torch; x=torch.ones(1); print('CPU check:', (x+1).item()); print('PyTorch:', torch.__version__)", label='cpu-check')
READY = True
print('Local CPU setup complete (float32).')


## Samples

The experiment contains 984 completions: 164 problems × 2 conditions × 3 runs. Existing samples are reused when resuming.

In [ ]:
if not globals().get('READY', False):
    raise RuntimeError('Setup has not completed.')
run('scripts/attribute_models.py', 'generate', '--config', CONFIG, '--output', RUN_DIR, label='attribution-generation')


## Results

Tables and plots below are computed from saved samples.

In [ ]:
if not globals().get('READY', False):
    raise RuntimeError('Complete local setup before attribution.')
run('scripts/attribute_models.py', 'attribute', '--config', CONFIG, '--output', RUN_DIR, label='attribution-attribution')
summary = json.loads((RUN_DIR / 'analysis/summary.json').read_text())
table = ['| Measurement | Result |', '|---|---|']
for key in ['watermarked_samples', 'clean_samples', 'unscored_samples', 'forced_accuracy',
            'thresholded_accuracy', 'attribution_coverage', 'clean_false_attribution_rate']:
    value = summary[key]
    shown = 'unavailable' if value is None else f'{value:.3f}' if isinstance(value, float) else str(value)
    table.append(f"| {key.replace('_', ' ')} | {shown} |")
display(Markdown('\n'.join(table)))
display(Image(filename=str(RUN_DIR / 'analysis/confusion.png')))
display(Markdown('**Recorded result:** every sample was scored against all three keys. '
    'Forced accuracy uses the largest Z; thresholded attribution also requires Z > 4. '
    'Ties and missing scores remain unknown. Unknown watermarked samples count against accuracy. '
    'These results identify registered keys, with models inferred from the key assignment. '
    'The maximum-score threshold is exploratory and has not been calibrated.'))

## Saved experiment

The archive contains samples, manifests, tables, and plots.

In [ ]:
if not any(RUN_DIR.glob('*/generations.jsonl')):
    print('No samples saved yet.')
else:
    EXPORT = shutil.make_archive(str(PROJECT / 'runs/local/humaneval/codeguard-attribution-results'), 'zip', RUN_DIR.parent, RUN_DIR.name)
    print('Local backup:', EXPORT)
